In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
import time
from tqdm import tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = "/kaggle/input/datasets/mdrifaturrahman33/levir-cd/LEVIR CD"

SHOW_PROGRESS = False
imgSize = 256
batch_size = 8
num_epochs =80
learning_rate = 1e-4

print(f"Device : {device}")
print(f"Image Size : {imgSize}")
print(f"Batch Size : {batch_size}")
print(f"Epochs : {num_epochs}")
print(f"Learning Rate : {learning_rate}")

Device : cuda
Image Size : 256
Batch Size : 8
Epochs : 80
Learning Rate : 0.0001


In [3]:
class ChangeDataset(Dataset):
    def __init__(self, root_dir, augment=False):
        self.root_dir = root_dir
        self.augment  = augment

        self.A = sorted(os.listdir(os.path.join(root_dir, "A")))
        self.B = sorted(os.listdir(os.path.join(root_dir, "B")))
        self.Y = sorted(os.listdir(os.path.join(root_dir, "label")))

        # FIX: Separate ToTensor+Normalize from spatial augmentations
        # Augmentation is done on PIL images BEFORE tensor conversion
        # so rotation + flip are applied jointly on img1, img2, and mask
        self.to_tensor_img = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225])
        ])
        self.to_tensor_mask = transforms.ToTensor()

    def __len__(self):
        return len(self.A)

    def __getitem__(self, idx):
        img1 = Image.open(os.path.join(self.root_dir, "A", self.A[idx])).convert("RGB")
        img2 = Image.open(os.path.join(self.root_dir, "B", self.B[idx])).convert("RGB")
        mask = Image.open(os.path.join(self.root_dir, "label", self.Y[idx])).convert("L")

        # Resize on PIL (avoids bilinear artifact on already-normalized tensors)
        img1 = img1.resize((imgSize, imgSize), Image.BILINEAR)
        img2 = img2.resize((imgSize, imgSize), Image.BILINEAR)
        mask = mask.resize((imgSize, imgSize), Image.NEAREST)

        if self.augment:
            # Random horizontal flip — applied jointly to both images + mask
            if torch.rand(1) < 0.5:
                img1 = TF.hflip(img1)
                img2 = TF.hflip(img2)
                mask = TF.hflip(mask)

            # Random vertical flip
            if torch.rand(1) < 0.5:
                img1 = TF.vflip(img1)
                img2 = TF.vflip(img2)
                mask = TF.vflip(mask)

            # FIX: Added random rotation (not in original)
            if torch.rand(1) < 0.3:
                angle = transforms.RandomRotation.get_params([-30, 30])
                img1 = TF.rotate(img1, angle, interpolation=InterpolationMode.BILINEAR)
                img2 = TF.rotate(img2, angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)

            # FIX: Color jitter on images only (independent, simulates sensor variation)
            if torch.rand(1) < 0.4:
                brightness = torch.empty(1).uniform_(0.75, 1.25).item()
                contrast   = torch.empty(1).uniform_(0.75, 1.25).item()
                img1 = TF.adjust_brightness(img1, brightness)
                img1 = TF.adjust_contrast(img1, contrast)
                img2 = TF.adjust_brightness(img2, torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_contrast(img2, torch.empty(1).uniform_(0.75, 1.25).item())

        img1 = self.to_tensor_img(img1)
        img2 = self.to_tensor_img(img2)
        mask = self.to_tensor_mask(mask)
        mask = (mask > 0.5).float()

        return img1, img2, mask

In [4]:
ROOT = "/kaggle/input/datasets/mdrifaturrahman33/levir-cd/LEVIR CD"

# FIX: augment=True only for training split
train_dataset = ChangeDataset(os.path.join(ROOT, "train"), augment=True)
val_dataset   = ChangeDataset(os.path.join(ROOT, "val"),   augment=False)
test_dataset  = ChangeDataset(os.path.join(ROOT, "test"),  augment=False)

# FIX: num_workers=2 for faster loading; pin_memory for GPU speed
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)

print("Train samples :", len(train_dataset))
print("Val samples   :", len(val_dataset))
print("Test samples  :", len(test_dataset))

Train samples : 445
Val samples   : 64
Test samples  : 128


In [5]:
# FIX: Decoder now uses concat skip (richer than add) + Conv-BN-ReLU to fuse
class DecoderBlock(nn.Module):
    """
    Upsample → concat skip → Conv-BN-ReLU
    skip is the per-scale DIFFERENCE |f1[i] - f2[i]| (not T1-only as before)
    """
    def __init__(self, in_ch, out_ch, skip_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)   # skip = |f1[i] - f2[i]|
        return self.conv(x)

In [6]:
# FIX: Entire model wrapped in one nn.Module — cleaner, no global loose variables
class ChangeDetectionNet(nn.Module):
    """
    Architecture:
      Shared ResNet-18 Siamese Encoder
      → Cross-Attention at bottleneck (Q=T1, K=V=T2)
      → Correct diff = |f1_s4 - attn_map|      [FIX 1]
      → Decoder with per-scale diff skips        [FIX 2]
      → Head without BatchNorm on final logit    [FIX 3]
    """
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Shared encoder stages (weights are shared via single encoder call)
        self.stage0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)  # → [B,64,128,128]
        self.stage1 = nn.Sequential(backbone.maxpool, backbone.layer1)             # → [B,64,64,64]
        self.stage2 = backbone.layer2                                               # → [B,128,32,32]
        self.stage3 = backbone.layer3                                               # → [B,256,16,16]
        self.stage4 = backbone.layer4                                               # → [B,512,8,8]

        # Cross-attention at bottleneck (Q=f1, K=V=f2)
        self.cross_attn = nn.MultiheadAttention(embed_dim=512, num_heads=8,
                                                 batch_first=True, dropout=0.1)

        # Decoder blocks: each takes upsampled x + diff skip from matching scale
        # channels:  in_ch → out_ch,  skip_ch = channels of diff at that scale
        self.dec4 = DecoderBlock(512, 256, skip_ch=256)   # skip: |f1[3]-f2[3]| = [B,256,16,16]
        self.dec3 = DecoderBlock(256, 128, skip_ch=128)   # skip: |f1[2]-f2[2]| = [B,128,32,32]
        self.dec2 = DecoderBlock(128,  64, skip_ch=64)    # skip: |f1[1]-f2[1]| = [B,64,64,64]
        self.dec1 = DecoderBlock( 64,  64, skip_ch=64)    # skip: |f1[0]-f2[0]| = [B,64,128,128]

        # FIX 3: Final head — ConvTranspose → ReLU → Conv1x1, NO BatchNorm on logit
        self.head = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # → [B,32,256,256]
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1)                        # → [B,1,256,256] raw logit
        )

    def encode(self, x):
        s0 = self.stage0(x)
        s1 = self.stage1(s0)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)
        return s0, s1, s2, s3, s4

    def forward(self, x1, x2):
        f1 = self.encode(x1)
        f2 = self.encode(x2)

        # --- Cross-attention at bottleneck ---
        B, C, H, W = f1[4].shape
        q = f1[4].flatten(2).permute(0, 2, 1)   # [B, HW, C]
        k = f2[4].flatten(2).permute(0, 2, 1)   # [B, HW, C]
        attn_out, _ = self.cross_attn(q, k, k)
        attn_map = attn_out.permute(0, 2, 1).reshape(B, C, H, W)

        # FIX 1: Correct difference — attn_map is attended F2, diff must be |F1 - F2_attended|
        # Original bug was: torch.abs(f1_s4 - 0.3 * attn_map) → wrong, made diff ≈ |F1|
        diff = torch.abs(f1[4] - attn_map)

        # FIX 2: Per-scale DIFFERENCE skip connections (not T1-only skips)
        x = self.dec4(diff, torch.abs(f1[3] - f2[3]))
        x = self.dec3(x,    torch.abs(f1[2] - f2[2]))
        x = self.dec2(x,    torch.abs(f1[1] - f2[1]))
        x = self.dec1(x,    torch.abs(f1[0] - f2[0]))

        return self.head(x)   # raw logit, no sigmoid here

In [7]:
torch.manual_seed(42)
np.random.seed(42)

model = ChangeDetectionNet().to(device)

# Shape sanity check
_a = torch.randn(2, 3, 256, 256).to(device)
_b = torch.randn(2, 3, 256, 256).to(device)
_out = model(_a, _b)
print("Output shape :", _out.shape)          # Expect [2, 1, 256, 256]
print("Output max   :", _out.max().item())   # Should be small — no longer 1036

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {total_params / 1e6:.2f}M")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s]


Output shape : torch.Size([2, 1, 256, 256])
Output max   : 0.1829591691493988
Trainable params: 14.56M


loss fn

In [8]:
# FIX 4: pos_weight handles ~95% no-change class imbalance in LEVIR-CD
# Without this, model learns to predict "no change" everywhere (high acc, near-zero recall)
POS_WEIGHT = 5.0   # tune between 3.0–8.0 based on dataset change ratio
bce = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT]).to(device)
)

def dice_loss(pred, target, smooth=1e-6):
    pred   = torch.sigmoid(pred)
    pred   = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def criterion(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

**Feature Difference**

In [9]:
def compute_metrics(pred, target):
    pred   = torch.sigmoid(pred)
    pred   = (pred > 0.5).float()
    tp = (pred  *  target      ).sum().float()
    tn = ((1-pred) * (1-target)).sum().float()
    fp = (pred  * (1-target)   ).sum().float()
    fn = ((1-pred) *  target   ).sum().float()
    return tp, tn, fp, fn

def finalize_metrics(tp, tn, fp, fn):
    eps  = 1e-8
    acc  = (tp + tn) / (tp + tn + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    iou  = tp / (tp + fp + fn + eps)   # FIX: Added IoU — required for paper comparison
    return {
        "acc" : acc.item(),
        "prec": prec.item(),
        "rec" : rec.item(),
        "f1"  : f1.item(),
        "iou" : iou.item()
    }

In [10]:
# FIX: AdamW (weight_decay regularization) instead of plain Adam
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=1e-4
)

# FIX: eta_min prevents LR from hitting zero completely at final epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

Stage 1 — Dual Temporal Input
Input Image T1
Input Image T2

Stage 2 — Shared Siamese Encoder (ResNet18 Backbone)
T1 → ResNet18 Encoder → F1
T2 → ResNet18 Encoder → F2

Output:
F1 = [1,512,8,8]
F2 = [1,512,8,8]

This stage corresponds to Siamese deep feature extraction used in early Siamese change detection papers and urban Siamese frameworks

Stage 3 — Tokenization for Transformer Attention
F1 → flatten → tokens
F2 → flatten → tokens

Shape:
[1,512,8,8] → [1,64,512]

Stage 4 — Cross Attention Fusion
Q = F1
K = F2
V = F2

Cross-attention:
Attention(F1,F2)
Output:
attn_out = [1,64,512]

Stage 5 — Attention Map Reconstruction
[1,64,512] → [1,512,8,8]

Result:
attn_map

Stage 6 — Difference Generation
(Current implied stage)
diff = |F1 - attn_map|
This is your current feature difference map.
Output:
diff = [1,512,8,8]


Stage 7 — Decoder Level 1
512 → 256
8×8 → 16×16

Output:
[1,256,16,16]


Stage 8 — Decoder Level 2
256 → 128
16×16 → 32×32
Output:
[1,128,32,32]
Stage 9 — Decoder Level 3
128 → 64
32×32 → 64×64
Stage 10 — Decoder Level 4
64 → 32
64×64 → 128×128
Stage 11 — Decoder Level 5 (Final Pixel Change Map)
32 → 1
128×128 → 256×256

Final:
Change Map = [1,1,256,256]

Stage 12 — Sigmoid Binary Pixel Probability
0 = no change
1 = change
Output values:
0 to 1 probability map
Full Current Architecture in One Line
T1,T2
→ Siamese Encoder
→ Cross Attention
→ Difference Map
→ Decoder Stack
→ Pixel Change Map
Current Architecture Position Relative to Full Research Architecture

You have completed:
✅ Siamese backbone
✅ Transformer fusion
✅ Pixel decoder

In [11]:
history = {
    "train_loss": [], "val_loss"  : [],
    "train_acc" : [], "val_acc"   : [],
    "train_prec": [], "val_prec"  : [],
    "train_rec" : [], "val_rec"   : [],
    "train_f1"  : [], "val_f1"    : [],
    "train_iou" : [], "val_iou"   : [],
}
best_val_f1 = 0.0
print("Training Started\n")
print(
    f"{'Ep':>3} | {'Time':>6} | "
    f"{'TrLoss':>7} | {'TrAcc':>6} | {'TrPrec':>7} | {'TrRec':>6} | {'TrF1':>6} | {'TrIoU':>7} | "
    f"{'VlLoss':>7} | {'VlAcc':>6} | {'VlPrec':>7} | {'VlRec':>6} | {'VlF1':>6} | {'VlIoU':>7} | {'LR':>9}"
)
print("─" * 140)

for epoch in range(num_epochs):
    start_time = time.time()

    # ──────────────── TRAIN ────────────────
    model.train()
    train_loss = 0.0
    tr_tp = tr_tn = tr_fp = tr_fn = 0

    for img1, img2, mask in tqdm(train_loader, leave=False, disable=not SHOW_PROGRESS):
        img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)

        pred = model(img1, img2)
        loss = criterion(pred, mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        tp, tn, fp, fn = compute_metrics(pred.detach(), mask)
        tr_tp += tp; tr_tn += tn; tr_fp += fp; tr_fn += fn

    # ──────────────── VALIDATION ────────────────
    model.eval()
    val_loss = 0.0
    v_tp = v_tn = v_fp = v_fn = 0

    with torch.no_grad():
        for img1, img2, mask in val_loader:
            img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
            pred = model(img1, img2)
            val_loss += criterion(pred, mask).item()
            tp, tn, fp, fn = compute_metrics(pred, mask)
            v_tp += tp; v_tn += tn; v_fp += fp; v_fn += fn

    # ──────────────── FINALIZE ────────────────
    tr_m  = finalize_metrics(tr_tp, tr_tn, tr_fp, tr_fn)
    val_m = finalize_metrics(v_tp,  v_tn,  v_fp,  v_fn)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # ──────────────── LOG TO HISTORY ────────────────
    history["train_loss"].append(train_loss / len(train_loader))
    history["val_loss"  ].append(val_loss   / len(val_loader))
    history["train_acc" ].append(tr_m["acc"])
    history["val_acc"   ].append(val_m["acc"])
    history["train_prec"].append(tr_m["prec"])
    history["val_prec"  ].append(val_m["prec"])
    history["train_rec" ].append(tr_m["rec"])
    history["val_rec"   ].append(val_m["rec"])
    history["train_f1"  ].append(tr_m["f1"])
    history["val_f1"    ].append(val_m["f1"])
    history["train_iou" ].append(tr_m["iou"])
    history["val_iou"   ].append(val_m["iou"])

    # ──────────────── SAVE BEST ────────────────
    if val_m["f1"] > best_val_f1:
        best_val_f1 = val_m["f1"]
        torch.save(model.state_dict(), "best_model_stage1_fixed.pth")
        saved_marker = " ✓"
    else:
        saved_marker = ""

    m, s = divmod(int(time.time() - start_time), 60)

    # ──────────────── PRINT ALL METRICS ────────────────
    print(
        f"{epoch+1:3d} | {m:2d}m {s:2d}s | "
        f"{history['train_loss'][-1]:7.4f} | "
        f"{tr_m['acc']:6.4f} | {tr_m['prec']:7.4f} | {tr_m['rec']:6.4f} | "
        f"{tr_m['f1']:6.4f} | {tr_m['iou']:7.4f} | "
        f"{history['val_loss'][-1]:7.4f} | "
        f"{val_m['acc']:6.4f} | {val_m['prec']:7.4f} | {val_m['rec']:6.4f} | "
        f"{val_m['f1']:6.4f} | {val_m['iou']:7.4f} | "
        f"{current_lr:.3e}"
        f"{saved_marker}"
    )

    # ──────────────── SEPARATOR every 10 epochs ────────────────
    if (epoch + 1) % 10 == 0:
        print("─" * 140)

print(f"\nBest Val F1 : {best_val_f1:.4f}")

Training Started

 Ep |   Time |  TrLoss |  TrAcc |  TrPrec |  TrRec |   TrF1 |   TrIoU |  VlLoss |  VlAcc |  VlPrec |  VlRec |   VlF1 |   VlIoU |        LR
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1 |  0m 37s |  1.5786 | 0.9387 |  0.2706 | 0.2205 | 0.2430 |  0.1383 |  1.4055 | 0.9362 |  0.3407 | 0.5573 | 0.4229 |  0.2681 | 9.996e-05 ✓
  2 |  0m 26s |  1.4021 | 0.9338 |  0.3702 | 0.6922 | 0.4824 |  0.3179 |  1.3058 | 0.9326 |  0.3695 | 0.8598 | 0.5169 |  0.3485 | 9.985e-05 ✓
  3 |  0m 27s |  1.2398 | 0.9442 |  0.4334 | 0.8181 | 0.5666 |  0.3953 |  1.1779 | 0.9222 |  0.3412 | 0.9162 | 0.4972 |  0.3309 | 9.966e-05
  4 |  0m 27s |  1.1002 | 0.9494 |  0.4630 | 0.8306 | 0.5946 |  0.4230 |  1.0068 | 0.9652 |  0.5615 | 0.7801 | 0.6530 |  0.4848 | 9.939e-05 ✓
  5 |  0m 26s |  0.9657 | 0.9544 |  0.4936 | 0.8499 | 0.6245 |  0.4541 |  0.9033 | 0.9591 |  0.5076 | 0.8627 | 0.6391 |  0.4697 | 9.905e

In [12]:
model.load_state_dict(torch.load("best_model_stage1_fixed.pth"))
model.eval()

test_loss = 0.0
t_tp = t_tn = t_fp = t_fn = 0

with torch.no_grad():
    for img1, img2, mask in test_loader:
        img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
        pred  = model(img1, img2)
        test_loss += criterion(pred, mask).item()
        tp, tn, fp, fn = compute_metrics(pred, mask)
        t_tp += tp; t_tn += tn; t_fp += fp; t_fn += fn

test_m = finalize_metrics(t_tp, t_tn, t_fp, t_fn)

print("\n" + "═" * 50)
print("          TEST RESULTS")
print("═" * 50)
print(f"  Loss      : {test_loss / len(test_loader):.4f}")
print("─" * 50)
print(f"  Accuracy  : {test_m['acc']:.4f}")
print(f"  Precision : {test_m['prec']:.4f}")
print(f"  Recall    : {test_m['rec']:.4f}")
print(f"  F1 Score  : {test_m['f1']:.4f}")
print(f"  IoU       : {test_m['iou']:.4f}")
print("═" * 50)


══════════════════════════════════════════════════
          TEST RESULTS
══════════════════════════════════════════════════
  Loss      : 0.4698
──────────────────────────────────────────────────
  Accuracy  : 0.9769
  Precision : 0.7563
  Recall    : 0.8054
  F1 Score  : 0.7801
  IoU       : 0.6394
══════════════════════════════════════════════════
